# LLM / RAG Tutor (Beginner → Expert)

**Run the notebook top to bottom** so every cell has what it needs from the cells above.

Then type a technical question into `question` (in the “Your question” section) and run the **last** code cell to see the tutor’s answer.

- **Default behavior**: concise answer with clear structure
- **Levels**: `beginner`, `intermediate`, `advanced`, `expert`
- **Focus areas**: LLMs, transformers, embeddings/vectors, vector DBs, RAG, evaluation, fine-tuning, prompting, agents

Optional environment variables: `OPENAI_API_KEY`, `OPENAI_MODEL`, `OLLAMA_HOST`, `OLLAMA_MODEL`, `TUTOR_PROVIDER` (`openai` | `ollama`), `TUTOR_LEVEL`.

Tip: Increase `level` or ask for an example, analogy, derivation, or code in your question.

### Imports

In [ ]:
import os
from typing import Literal, Optional

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

### Configuration (models & defaults)

The next cell defines **which models** to use and **default tutor settings**.

In [ ]:
from curses import echo

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2")

Provider = Literal["openai", "ollama"]
Level = Literal["beginner", "intermediate", "advanced", "expert"]

def _parse_provider(raw: str | None) -> Provider:
    """Map TUTOR_PROVIDER env to a supported backend; invalid values fall back to openai."""
    key = (raw or "openai").strip().lower()
    return "ollama" if key == "ollama" else "openai"

def _parse_level(raw: str | None) -> Level:
    """Map TUTOR_LEVEL env to a supported depth; invalid values fall back to intermediate."""
    key = (raw or "intermediate").strip().lower()
    allowed: tuple[Level, ...] = ("beginner", "intermediate", "advanced", "expert")
    return key if key in allowed else "intermediate"

DEFAULT_PROVIDER: Provider = _parse_provider(os.getenv("TUTOR_PROVIDER"))
DEFAULT_LEVEL: Level = _parse_level(os.getenv("TUTOR_LEVEL"))

### Tutor setup
The next cell prepares **runtime setup** and the **tutor’s behavior**. All later cells can call the API with consistent instructions.

In [ ]:
load_dotenv()

_openai_client: Optional[OpenAI] = None


def get_openai_client() -> OpenAI:
    """Return a shared OpenAI client.

    Requires `OPENAI_API_KEY` to be set (either in your shell env or in a `.env` file).
    """

    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError(
            "Missing OPENAI_API_KEY.\n\n"
            "Fix options:\n"
            "- Set OPENAI_API_KEY in your environment or in a .env file, then re-run this cell\n"
            "- OR switch provider='ollama' to use a local model"
        )

    global _openai_client
    if _openai_client is None:
        _openai_client = OpenAI(api_key=api_key)
    return _openai_client


OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://localhost:11434")

TUTOR_SYSTEM = """
You are an expert tutor in LLM machine learning and modern AI.

Your job: teach students from beginner to expert, quickly.

Response rules:
- Be concise but crystal-clear.
- Use a predictable structure:
  1) One-sentence answer
  2) Key ideas (bullets)
  3) Mini example (if helpful; keep it short)
  4) Common pitfalls (1–3 bullets)
  5) Next step (1 actionable suggestion)
- Match depth to the requested LEVEL.
- If the question is ambiguous, state the most likely interpretation and proceed.
- When explaining RAG/vectors: distinguish embedding model vs vector index vs retriever vs reranker vs generator.
- Avoid long preambles, filler, or marketing language.
""".strip()

LEVEL_GUIDANCE: dict[Level, str] = {
    "beginner": "Use plain language, define jargon, 0–1 equations, 0–1 short example.",
    "intermediate": "Assume basic ML knowledge. Include mechanisms and trade-offs.",
    "advanced": "Include failure modes, evaluation, and design choices. Use precise terminology.",
    "expert": "Be very technical: objective functions, scaling laws intuitions, system constraints, and edge cases.",
}

### Your question & controls

The next cell is the main place **you edit**:

- **`question`**: what you want the tutor to explain (LLMs, RAG, vectors, etc.).
- **`provider`**: `openai` (cloud) or `ollama` (your machine).
  - If you see an error about `OPENAI_API_KEY`, either set that key (shell or `.env`) or switch to `ollama`.
- **`level`**: how deep the explanation should go.

**Result:** those values are passed into `ask_tutor` when you run the last cell.

In [ ]:
question = """Explain RAG end-to-end: embeddings, vector DB, retrieval, reranking, context window, and generation."""

provider: Provider = DEFAULT_PROVIDER
level: Level = DEFAULT_LEVEL

### The `ask_tutor` function

The next cell **defines** `ask_tutor(...)`. It does **not** call the model yet.

- Builds a **user message** that includes the difficulty (`level`) and your `question`.

**Return value:** plain text (markdown-formatted string) that the last cell will display.

In [ ]:
def ask_tutor(
    q: str,
    *,
    provider: Provider = DEFAULT_PROVIDER,
    level: Level = DEFAULT_LEVEL,
    model: Optional[str] = None,
) -> str:
    """Return the tutor's answer text for question `q`."""
    q = (q or "").strip()
    if not q:
        raise ValueError("Question is empty")

    level_note = LEVEL_GUIDANCE[level]
    user_msg = f"""LEVEL={level}\n\n{level_note}\n\nQuestion: {q}"""

    if provider == "openai":
        client = get_openai_client()
        resp = client.chat.completions.create(
            model=model or OPENAI_MODEL,
            messages=[
                {"role": "system", "content": TUTOR_SYSTEM},
                {"role": "user", "content": user_msg},
            ],
        )
        return (resp.choices[0].message.content or "").strip()

    if provider == "ollama":
        r = requests.post(
            f"{OLLAMA_HOST.rstrip('/')}/api/chat",
            json={
                "model": model or OLLAMA_MODEL,
                "stream": False,
                "messages": [
                    {"role": "system", "content": TUTOR_SYSTEM},
                    {"role": "user", "content": user_msg},
                ],
            },
            timeout=120,
        )
        r.raise_for_status()
        data = r.json()
        return (data.get("message", {}) or {}).get("content", "").strip()

    raise ValueError(f"Unknown provider: {provider}")

### Get the answer

The next cell **calls the tutor** with your `question`, `provider`, and `level`, then **renders** the reply as markdown below the cell.

If something fails: check `OPENAI_API_KEY` for OpenAI, or that Ollama is running and the model name exists for the Ollama path.

In [ ]:
try:
    answer = ask_tutor(question, provider=provider, level=level)
    display(Markdown(answer))
except Exception as e:
    display(
        Markdown(
            "## Setup issue\n"
            f"**{type(e).__name__}**: {e}\n\n"
            "### Quick fixes\n"
            "- For **OpenAI**: set `OPENAI_API_KEY` (shell env or `.env`), then rerun the notebook.\n"
            "- For **Ollama**: set `provider = 'ollama'`, ensure Ollama is running, and the model name exists.\n"
        )
    )